# NBA Draft Bust/Star — Exploratory Data Analysis

This notebook walks through the processed dataset and produces the five EDA figures used in the project write-up:

1. Class distribution bar chart
2. Draft pick number vs WS scatter colored by label
3. Boxplots of key features by class
4. Correlation heatmap of features
5. Missing data heatmap

All figures are also saved to `outputs/figures/` by the standalone module `src/eda.py`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
%matplotlib inline

df = pd.read_csv(ROOT / 'data' / 'processed' / 'nba_draft_processed.csv')
print('shape:', df.shape)
df.head()

## Class distribution

In [ ]:
CLASS_ORDER = ['Bust', 'Solid', 'Star']
CLASS_COLORS = {'Bust': '#d95f02', 'Solid': '#7570b3', 'Star': '#1b9e77'}

counts = df['career_label'].value_counts().reindex(CLASS_ORDER)
print(counts)
print('\npercentages:')
print((counts / counts.sum() * 100).round(1))

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
bars = ax.bar(counts.index, counts.values, color=[CLASS_COLORS[c] for c in counts.index])
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, val, f'{int(val)}\n({val/counts.sum()*100:.1f}%)',
            ha='center', va='bottom', fontsize=10)
ax.set_title('Class distribution — career_label')
ax.set_ylabel('Players')
ax.set_ylim(0, counts.values.max()*1.18)
plt.show()

## Draft pick vs career WS

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for cls in CLASS_ORDER:
    sub = df[df['career_label']==cls]
    ax.scatter(sub['draft_pick'], sub['career_ws'], alpha=0.55, s=20,
               color=CLASS_COLORS[cls], label=cls)
ax.axhline(50, ls='--', color='gray', alpha=0.5, label='Star threshold (WS=50)')
ax.axhline(15, ls=':', color='gray', alpha=0.5, label='Solid threshold (WS=15)')
ax.set_xlabel('Draft pick')
ax.set_ylabel('Career Win Shares')
ax.set_title('Draft pick vs career WS')
ax.legend()
plt.show()

Top picks (low pick number) dominate the upper region (high WS). The relationship is strongly non-linear and noisy — plenty of high picks bust and plenty of late picks succeed.

## Feature boxplots by class

In [ ]:
cols = ['draft_pick', 'height_no_shoes', 'wingspan', 'vertical_leap_max']
available = [c for c in cols if c in df.columns]
fig, axes = plt.subplots(1, len(available), figsize=(4*len(available), 4))
for ax, col in zip(axes, available):
    sns.boxplot(data=df, x='career_label', y=col, order=CLASS_ORDER, palette=CLASS_COLORS, ax=ax)
    ax.set_title(col)
    ax.set_xlabel('')
plt.suptitle('Feature distributions by career label', y=1.02)
plt.tight_layout()
plt.show()

## Correlation heatmap

In [ ]:
from features import NUMERIC_FEATURES
numeric = [c for c in NUMERIC_FEATURES if c in df.columns]
corr = df[numeric].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax, annot_kws={'size':8})
ax.set_title('Numeric feature correlations')
plt.xticks(rotation=45, ha='right')
plt.show()

Height, weight, wingspan, standing reach cluster tightly — they're physically related. Vertical leap is roughly inverse to weight/height (smaller players jump higher).

## Missing data heatmap

In [ ]:
cols = numeric + (['position'] if 'position' in df.columns else [])
mask = df[cols].isna().astype(int)
fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(mask, cbar=False, yticklabels=False, cmap=['#f0f0f0', '#d95f02'], ax=ax)
ax.set_title('Missing data (orange = missing)')
plt.xticks(rotation=45, ha='right')
plt.show()

print('\nMissingness by feature (%):')
print((mask.mean()*100).round(1).sort_values(ascending=False))

Combine measurements are missing for pre-2000 picks (no combine data existed) and for post-2000 players who skipped the combine. The model handles this with median imputation fit on training folds only.